
# Bonus: Inferencing with a local fine-tuned model using Ollama

This is completely optional, and we do not plan to cover this in the lab. Try this if you have absolutely nothing else to do :)

## Installing OLLAMA on Windows and MacOS

Follow the instructions here provided here :https://github.com/ollama/ollama


In [23]:
# Verify Installation
!ollama --version

# Start the Ollama server
# Note: The server will run in the background

# On Windows
# The always-on API lets you integrate Ollama into your projects with ease. It serves requests on http://localhost:11434

# On MacOS, you can start the server with the following command:
!brew services start ollama

### Run a fine-tuned model locally using Ollama

NB: this assumes you have a fine-tuned model in the folder 'output_dir/mlflow_model_folder/data/model'

### Create a modelfile

In [24]:
# Step 1: Start the Ollama server with the fine-tuned model
# In the sample Modelfile below, replace 'output_dir/mlflow_model_folder/data/model' with the path to your fine-tuned model within the repo
# change the model name to your model name
!ollama create llama32-instruct-ft -f Modelfile


In [29]:
import requests
import json
from textwrap import dedent
from typing import List, Tuple, Dict, Any

# Post-processing function: formats the model output
def postprocess_output(raw_output: str) -> Dict[str, Any]:
    """
    Converts model output into a structured dictionary format:
    {"function_name": <function_name>, "parameters": {"param1": "value1", ...}}
    """
    try:
        parsed_output = json.loads(raw_output)  # Assuming the model returns JSON-like text
        if isinstance(parsed_output, dict) and "function_name" in parsed_output:
            return parsed_output  # Valid structured output
    except json.JSONDecodeError:
        pass

    # Default fallback if model response is not well-formed
    return {"function_name": "unknown_function", "parameters": {"raw_response": raw_output}}

# Function to calculate metrics
def calculate_metrics(response_json: Dict[str, Any]) -> Dict[str, int]:
    """
    Extracts and returns metrics from the response JSON.
    """
    return response_json.get('usage', {})

# Run inference on a list of queries
def run_inference(queries: List[str], model: str, system_prompt: str) -> List[Tuple[Dict[str, Any], Dict[str, int]]]:
    """
    Runs inference on multiple queries.

    Returns:
        List of tuples: [(formatted_answer, metrics), ...]
    """
    url = "http://localhost:11434/v1/chat/completions"
    headers = {
        "Content-Type": "application/json"
    }
    results = []

    for query in queries:
        data = {
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": query}
            ]
        }

        response = requests.post(url, headers=headers, data=json.dumps(data))
        response_json = response.json()

        # Post-process the model output
        raw_output = response_json['choices'][0]['message']['content']
        formatted_output = postprocess_output(raw_output)

        # Calculate metrics
        metrics = calculate_metrics(response_json)

        results.append((formatted_output, metrics))

    return results

# Example usage
queries = [
    "What is the capital of France?",
    "Who is the president of the United States?"
]

model = "llama32-instruct-ft" # Change this to your model name
system_prompt = SYSTEM_PROMPT = dedent(
    """
    You are an in car virtual assistant that maps user's inputs to the corresponding function call in the vehicle. You must respond with only a JSON object matching the following schema:

    {
        "name": <name of the function>,
        "arguments": {
            <arg1_name>: <arg1_value>,
            <arg2_name>: <arg2_value>,
            ...
        }
    }

    Important: DO NOT include any other information in the response.
    """
)

responses_with_metrics = run_inference(queries, model, system_prompt)
for response, metrics in responses_with_metrics:
    print(f"Model Response: {json.dumps(response, indent=2)}")
    print(f"Metrics: {metrics}")